# Indonesian ASTE experiment notebook

This notebook is the researcher-facing entry point for running and inspecting the approved ASTE experiments. It delegates training, checkpoint selection, held-out evaluation, and provenance capture to the existing `aste-run` pipeline; model and evaluator code stay in `src/aste_repro`.

The active execution gate is the 20-run HoASA + EMCGCN baseline (four pinned encoders × seeds `0, 1, 2, 3, 52`). Every optimizer group uses the configured `2e-5` learning rate. Earlier runs with PyABSA's `1e-3` graph/classifier rate remain archived under their original identities and are excluded from this protocol. The notebook starts no jobs unless explicitly enabled and refuses to duplicate an active-protocol run or overlap the active queue. CASA remains blocked until human-adjudicated, sentiment-bearing ASTE gold is frozen; SSGCN follows the baseline/source gate.

Only `complete` manifests with test metrics and the active experiment protocol ID count as results. Smoke, failed, interrupted, running, superseded-protocol, and absent runs are not counted.


## 1. Load the frozen study configuration

Run this from the repository root, `sentiment-analysis/`, or this notebook's directory. The matrix JSON is the source of truth for encoder revisions, seeds, targets, training controls, and the experiment protocol ID.


In [1]:
from pathlib import Path
import json
import os
import shutil
import statistics
import subprocess
import time
from datetime import datetime, timezone


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs" / "experiment-matrix.json").is_file():
            return candidate
        nested = candidate / "sentiment-analysis"
        if (nested / "configs" / "experiment-matrix.json").is_file():
            return nested
    raise FileNotFoundError("Could not find sentiment-analysis/configs/experiment-matrix.json")


PROJECT = find_project_root()
REPO_ROOT = PROJECT.parent
RUNS = PROJECT / "runs"
REPORT = PROJECT / "results" / "aste-reproduction-report.md"
MATRIX_PATH = PROJECT / "configs" / "experiment-matrix.json"
QUEUE_MARKER = RUNS / "hoasa-emcgcn-baseline-queue.status.json"
MATRIX = json.loads(MATRIX_PATH.read_text(encoding="utf-8"))
UV = shutil.which("uv")
if UV is None:
    candidates = [Path.home() / ".local" / "bin" / "uv.exe", Path.home() / ".local" / "bin" / "uv"]
    UV = str(next((path for path in candidates if path.exists()), "uv"))

print(f"Project: {PROJECT}")
print(f"Run artifacts: {RUNS}")
print(f"Aggregate report: {REPORT}")
print(f"Experiment protocol: {MATRIX['experiment_protocol_id']}")
print(f"Declared seeds: {MATRIX['seed_protocol']['seeds']}")
print(f"Configured learning rate for every optimizer group: {MATRIX['training']['learning_rate']}")
print(f"Training controls: {MATRIX['training']['epochs']} epochs, batch size {MATRIX['training']['batch_size']}")
print(f"Pinned encoders: {', '.join(MATRIX['encoders'])}")


Project: C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis
Run artifacts: C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis\runs
Aggregate report: C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis\results\aste-reproduction-report.md
Experiment protocol: indonesian-aste-matrix-v2-uniform-learning-rate
Declared seeds: [0, 1, 2, 3, 52]
Configured learning rate for every optimizer group: 2e-05
Training controls: 10 epochs, batch size 16
Pinned encoders: mbert, indobert, xlmr, deberta_absa


## 2. Inspect run status and recorded test scores

This reads immutable run manifests and metrics written by the training pipeline. It does not evaluate a model or infer scores from logs.


In [2]:
def read_runs():
    records = []
    if not RUNS.exists():
        return records
    for manifest_path in sorted(RUNS.glob("*/manifest.json")):
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue  # a manifest may be in the middle of being written
        metrics_path = manifest_path.parent / "metrics.json"
        metrics = None
        if metrics_path.is_file():
            try:
                metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                pass
        test_f1 = None
        if manifest.get("status") == "complete" and metrics:
            value = metrics.get("test", {}).get("triplet", {}).get("f1")
            if value is not None:
                test_f1 = 100.0 * float(value)
        records.append({
            "run_id": manifest.get("run_id", manifest_path.parent.name),
            "protocol_id": manifest.get("experiment_protocol_id"),
            "dataset": manifest.get("dataset"),
            "architecture": manifest.get("architecture"),
            "encoder": manifest.get("encoder"),
            "seed": manifest.get("seed"),
            "status": manifest.get("status"),
            "test_triplet_f1_pct": test_f1,
            "run_dir": str(manifest_path.parent),
        })
    return records


def show_baseline_status():
    rows = [row for row in read_runs() if row["protocol_id"] == MATRIX["experiment_protocol_id"] and row["dataset"] == "hoasa" and row["architecture"] == "emcgcn"]
    rows.sort(key=lambda row: (row["encoder"] or "", row["seed"] if row["seed"] is not None else -1))
    if not rows:
        print("No HoASA/EMCGCN runs recorded for the active protocol yet.")
    else:
        for row in rows:
            score = "—" if row["test_triplet_f1_pct"] is None else f"{row['test_triplet_f1_pct']:.2f}%"
            print(f"{row['encoder']:14} seed={str(row['seed']):>2}  {row['status']:22}  test F1={score}  {row['run_id']}")
    return rows


baseline_status = show_baseline_status()
if QUEUE_MARKER.exists():
    queue = json.loads(QUEUE_MARKER.read_text(encoding="utf-8"))
    label = "active" if queue.get("status") == "running" else "last"
    print(f"\nBaseline queue {label} status: {queue.get('status', 'unknown')} (PID {queue.get('queue_pid')}): {queue.get('scope')}")


No HoASA/EMCGCN runs recorded for the active protocol yet.


## 3. Run a selected condition

Set `RUN_ONE_CONDITION = True` only when you intend to launch the selected full benchmark. The function delegates to `aste-run`, preserves the declared seed and source controls, and rejects duplicate complete/running conditions within the active protocol. Superseded runs do not satisfy the new protocol. It also refuses to overlap the active baseline queue.


In [3]:
def matching_runs(dataset, architecture, encoder, seed):
    return [row for row in read_runs()
            if row["protocol_id"] == MATRIX["experiment_protocol_id"]
            and row["dataset"] == dataset
            and row["architecture"] == architecture
            and row["encoder"] == encoder
            and int(row["seed"]) == int(seed)]


def assert_no_active_queue():
    if not QUEUE_MARKER.exists():
        return
    queue = json.loads(QUEUE_MARKER.read_text(encoding="utf-8"))
    if queue.get("status") != "running":
        return  # completed/failed markers are history, not active locks
    try:
        pid = int(queue.get("queue_pid"))
    except (TypeError, ValueError):
        raise RuntimeError("The queue marker says running but has no valid process ID.")
    if pid == os.getpid():
        return  # this notebook process owns the active queue
    if os.name == "nt":
        result = subprocess.run(["tasklist", "/FI", f"PID eq {pid}", "/NH"],
                                capture_output=True, text=True)
        if result.returncode == 0 and not any(
            line.strip().split() and line.strip().split()[0] == str(pid)
            for line in result.stdout.splitlines()
        ):
            return  # stale running marker; the next queue invocation will replace it
        if result.returncode != 0:
            raise RuntimeError("Could not verify the existing queue process; refusing to start another.")
    else:
        try:
            os.kill(pid, 0)
        except OSError:
            return  # stale running marker; the next queue invocation will replace it
    raise RuntimeError(
        "An HoASA EMCGCN benchmark queue is already active "
        f"(PID {pid}; scope: {queue.get('scope')}). "
        "Inspect run status above; do not start overlapping jobs."
    )


def refresh_report():
    command = [UV, "run", "--python", "3.10", "--project", str(PROJECT),
               "aste-report", "--output", str(REPORT)]
    result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    print(result.stdout.strip())


def run_condition(dataset, architecture, encoder, seed):
    assert_no_active_queue()
    existing = matching_runs(dataset, architecture, encoder, seed)
    if any(row["status"] == "running" for row in existing):
        raise RuntimeError("This condition already has a running benchmark manifest.")
    if any(row["status"] == "complete" for row in existing):
        raise RuntimeError("This seed already has a complete run; refusing to duplicate it.")
    command = [UV, "run", "--python", "3.10", "--project", str(PROJECT),
               "aste-run", "--dataset", dataset, "--architecture", architecture,
               "--encoder", encoder, "--seed", str(seed), "--purpose", "benchmark"]
    print("Launching:", " ".join(command))
    result = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
    if result.returncode:
        # Native process crashes can bypass training.py's exception handler and
        # leave a manifest marked running. Record the exit without inventing a score.
        failure = f"aste-run exited with status {result.returncode}; no benchmark score was recorded."
        for row in matching_runs(dataset, architecture, encoder, seed):
            if row["status"] != "running":
                continue
            manifest_path = Path(row["run_dir"]) / "manifest.json"
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest.get("status") == "running":
                manifest.update({
                    "status": "failed",
                    "completed_at": datetime.now(timezone.utc).isoformat(),
                    "failure_type": "ProcessExit",
                    "failure": failure,
                })
                manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(failure)
        if result.stderr:
            print("\n".join(result.stderr.splitlines()[-12:]))
        raise subprocess.CalledProcessError(result.returncode, command, result.stdout, result.stderr)
    refresh_report()
    completed = [row for row in matching_runs(dataset, architecture, encoder, seed)
                 if row["status"] == "complete" and row["test_triplet_f1_pct"] is not None]
    if completed:
        newest = max(completed, key=lambda row: (Path(row["run_dir"]) / "manifest.json").stat().st_mtime)
        print(f"Completed {newest['run_id']}: held-out triplet F1={newest['test_triplet_f1_pct']:.2f}%")
    return result.stdout


RUN_ONE_CONDITION = False  # Change to True before running the cell to launch the selection below.
SELECTED = {"dataset": "hoasa", "architecture": "emcgcn", "encoder": "mbert", "seed": 0}
if RUN_ONE_CONDITION:
    run_condition(**SELECTED)
else:
    print("Execution disabled. Set RUN_ONE_CONDITION = True to launch the selected benchmark.")


Execution disabled. Set RUN_ONE_CONDITION = True to launch the selected benchmark.


## 4. Continue missing HoASA + EMCGCN baseline seeds\n\nThis cell reruns the approved first gate under the corrected optimizer protocol: 20 full runs across four encoders and five seeds. It skips only active-protocol complete seeds, retries failed/interrupted active-protocol conditions as new immutable runs, and continues after a failed condition. Earlier runs with the 1e-3 graph/classifier learning rate do not count as completed seeds. Existing active queues are protected by their process ID. Expect multi-day CPU runtime; keep execution disabled while another queue is active.\n

In [4]:
BASELINE_CONDITIONS = [
    {"dataset": "hoasa", "architecture": "emcgcn", "encoder": encoder, "seed": seed}
    for encoder in MATRIX["encoders"]
    for seed in MATRIX["seed_protocol"]["seeds"]
]


def write_queue_status(status, **extra):
    record = {
        "status": status,
        "queue_pid": os.getpid(),
        "started_at": queue_started_at,
        "scope": queue_scope,
        "experiment_protocol_id": MATRIX["experiment_protocol_id"],
        "recovered_queue": previous_queue,
        **extra,
    }
    QUEUE_MARKER.write_text(json.dumps(record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def run_missing_baseline():
    global queue_started_at, queue_scope, previous_queue
    assert_no_active_queue()
    previous_queue = None
    if QUEUE_MARKER.exists():
        old = json.loads(QUEUE_MARKER.read_text(encoding="utf-8"))
        previous_queue = {key: old[key] for key in (
            "status", "queue_pid", "started_at", "completed_at", "failure_type", "failure"
        ) if key in old}
    queue_started_at = datetime.now(timezone.utc).isoformat()
    queue_scope = f"{MATRIX['experiment_protocol_id']}: hoasa/emcgcn seeds {MATRIX['seed_protocol']['seeds']} across {list(MATRIX['encoders'])}"
    write_queue_status("running")
    failures = []
    try:
        for condition in BASELINE_CONDITIONS:
            existing = matching_runs(**condition)
            if any(row["status"] == "running" for row in existing):
                failures.append({"condition": condition, "error": "matching run manifest is still marked running"})
                print(f"Cannot resume stale running manifest; inspect and recover first: {condition}")
                continue
            if any(row["status"] == "complete" for row in existing):
                print(f"Already complete; skip {condition}")
                continue
            try:
                run_condition(**condition)
            except Exception as error:
                failures.append({"condition": condition, "error": f"{type(error).__name__}: {error}"})
                print(f"Run failed; continuing to remaining conditions: {condition}")
        refresh_report()
    except KeyboardInterrupt as error:
        write_queue_status("interrupted", completed_at=datetime.now(timezone.utc).isoformat(),
                           failure_type=type(error).__name__, failure="queue interrupted before all conditions completed")
        raise
    except Exception as error:
        write_queue_status("failed", completed_at=datetime.now(timezone.utc).isoformat(),
                           failure_type=type(error).__name__, failure=str(error)[:1000])
        raise
    status = "complete_with_failures" if failures else "complete"
    write_queue_status(status, completed_at=datetime.now(timezone.utc).isoformat(), failures=failures)
    print(f"Baseline queue {status}: {len(failures)} failed condition(s).")


EXECUTE_REMAINING_BASELINE = os.environ.get("ASTE_EXECUTE_REMAINING_BASELINE") == "1"  # Process-local explicit opt-in.
if EXECUTE_REMAINING_BASELINE:
    run_missing_baseline()
else:
    print(f"Execution disabled; {len(BASELINE_CONDITIONS)} planned HoASA/EMCGCN seed runs.")


Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder mbert --seed 0 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'mbert', 'seed': 0}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder mbert --seed 1 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'mbert', 'seed': 1}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder mbert --seed 2 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'mbert', 'seed': 2}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder mbert --seed 3 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'mbert', 'seed': 3}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder mbert --seed 52 --purpose benchmark


...

aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'deberta_absa', 'seed': 1}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder deberta_absa --seed 2 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'deberta_absa', 'seed': 2}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder deberta_absa --seed 3 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'deberta_absa', 'seed': 3}
Launching: C:\Users\Lyrid\.local\bin\uv.EXE run --python 3.10 --project C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis aste-run --dataset hoasa --architecture emcgcn --encoder deberta_absa --seed 52 --purpose benchmark


aste-run exited with status 1; no benchmark score was recorded.
run failed: ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']
Run failed; continuing to remaining conditions: {'dataset': 'hoasa', 'architecture': 'emcgcn', 'encoder': 'deberta_absa', 'seed': 52}


wrote C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis\results\aste-reproduction-report.md (8611 characters)
Baseline queue complete_with_failures: 20 failed condition(s).


## 5. Seed-level results and descriptive statistics

Only complete runs with recorded held-out test metrics appear here. Scores are exact triplet micro-F1 percentages, matching the aggregate report.


In [5]:
def baseline_results():
    completed = [row for row in read_runs()
                 if row["protocol_id"] == MATRIX["experiment_protocol_id"]
                 and row["dataset"] == "hoasa"
                 and row["architecture"] == "emcgcn"
                 and row["status"] == "complete"
                 and row["test_triplet_f1_pct"] is not None]
    output = []
    for encoder, spec in MATRIX["encoders"].items():
        scores = sorted((row["seed"], row["test_triplet_f1_pct"])
                        for row in completed if row["encoder"] == encoder)
        values = [score for _, score in scores]
        output.append({
            "encoder": spec["display_name"],
            "n_complete": len(values),
            "seed_f1_pct": ", ".join(f"{seed}: {score:.2f}" for seed, score in scores) or "—",
            "mean_pct": round(statistics.mean(values), 2) if values else None,
            "sd_pct": round(statistics.stdev(values), 2) if len(values) > 1 else None,
            "thesis_target_pct": spec["target_mean_triplet_f1"],
        })
    return output


baseline_summary = baseline_results()
for row in baseline_summary:
    print(row)


{'encoder': 'mBERT', 'n_complete': 0, 'seed_f1_pct': '—', 'mean_pct': None, 'sd_pct': None, 'thesis_target_pct': 75.72}
{'encoder': 'IndoBERT', 'n_complete': 0, 'seed_f1_pct': '—', 'mean_pct': None, 'sd_pct': None, 'thesis_target_pct': 78.01}
{'encoder': 'XLM-RoBERTa', 'n_complete': 0, 'seed_f1_pct': '—', 'mean_pct': None, 'sd_pct': None, 'thesis_target_pct': 77.97}
{'encoder': 'DeBERTa-ABSA', 'n_complete': 0, 'seed_f1_pct': '—', 'mean_pct': None, 'sd_pct': None, 'thesis_target_pct': 80.79}


## 6. Refresh and open the aggregate report

The report retains the full declared matrix, marks blocked/unrun conditions, and computes current summaries only from complete manifests carrying the active protocol ID. It also shows prior superseded baseline results in a separate historical section. CASA remains unmeasured until its gold-data gate is resolved; partial or smoke runs never become scores.


In [6]:
refresh_report()
print(f"\nOpen the generated report at: {REPORT}")


wrote C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis\results\aste-reproduction-report.md (8611 characters)

Open the generated report at: C:\Users\Lyrid\Documents\Projects\research\sentiment-analysis\results\aste-reproduction-report.md
